In [1]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
REPO_DIR = "/workspace/18-4-2026"
EXP_DIR = os.path.join(REPO_DIR, "experiments", "2026", "15-4-2026")

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(EXP_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory so Path.cwd().parent resolves to experiment root
os.chdir(os.path.join(EXP_DIR, "notebooks"))

From https://github.com/safety-research/legibility
   5e5fb91..2843de3  main       -> origin/main


HEAD is now at 2843de3 Fix NB4 plot: clamp negative yerr values to 0


# NB5: Attention Pattern Analysis (Experiment A3)

**GPU notebook** (~1 hour). Reload G3, run forward pass with `output_attentions=True`
on legible vs illegible CoTs. Compare attention entropy per head per layer.

Hypothesis: legible CoTs have more concentrated attention patterns (focused on
specific reasoning steps), while illegible CoTs have more diffuse attention.

**Requires:** `results/classifications.json`, Step 1 generation logs

In [2]:
import sys
import json
import gc
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    join_cots_with_labels, load_model, bootstrap_ci_metric,
    PHASE2_RESULTS_DIR, LOCAL_MODELS,
)

PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"CUDA: {torch.cuda.is_available()}")

CUDA: True


In [3]:
# Load legible + illegible CoTs for G3
cots = join_cots_with_labels(
    labels=['REASONING_LEGIBLE', 'ILLEGIBLE'],
    generator_ids=['G3'],
)
cots = sorted(cots, key=lambda x: (x['sample_id'], x['epoch']))

legible = [c for c in cots if c['label'] == 'REASONING_LEGIBLE']
illegible = [c for c in cots if c['label'] == 'ILLEGIBLE']
print(f"Legible: {len(legible)}, Illegible: {len(illegible)}")

# Limit to manageable sample size (attention tensors are large)
MAX_SAMPLES = 50
legible_sample = legible[:MAX_SAMPLES]
illegible_sample = illegible[:MAX_SAMPLES]
print(f"Using: {len(legible_sample)} legible, {len(illegible_sample)} illegible")

Legible: 28, Illegible: 23
Using: 28 legible, 23 illegible


In [ ]:
# Check if attention entropy arrays already exist (skip model load if so)
_leg_entropy_path = PHASE2_RESULTS_DIR / 'legible_attention_entropy.npy'
_ill_entropy_path = PHASE2_RESULTS_DIR / 'illegible_attention_entropy.npy'
_entropy_cached = _leg_entropy_path.exists() and _ill_entropy_path.exists()

if _entropy_cached:
    print("CACHED: Attention entropy arrays exist, skipping G3 model load")
    model = tokenizer = None
    n_layers = n_heads = None
else:
    # Load G3 with eager attention (required for output_attentions=True)
    # SDPA doesn't support returning attention weights
    from transformers import AutoModelForCausalLM, AutoTokenizer
    hf_id = LOCAL_MODELS["G3"]["hf_id"]
    local_path = LOCAL_MODELS["G3"].get("local_path", hf_id)
    print(f"Loading {hf_id} with attn_implementation='eager'...")
    tokenizer = AutoTokenizer.from_pretrained(local_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        local_path,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="eager",
    )
    model.eval()
    n_layers = model.config.num_hidden_layers
    n_heads = model.config.num_attention_heads
    print(f"Layers: {n_layers}, Heads: {n_heads}")

In [5]:
if _entropy_cached:
    print(f"CACHED: Loading pre-computed attention entropy arrays")
    legible_entropy = np.load(_leg_entropy_path)
    illegible_entropy = np.load(_ill_entropy_path)
    print(f"Legible entropy shape: {legible_entropy.shape}")
    print(f"Illegible entropy shape: {illegible_entropy.shape}")
else:
    def compute_attention_entropy(model, tokenizer, texts, max_length=4096):
        """Compute per-head attention entropy for each text.
        
        Returns: array of shape (n_texts, n_layers, n_heads) with mean entropy per head.
        """
        device = next(model.parameters()).device
        all_entropies = []
        
        for text in tqdm(texts, desc="Computing attention entropy"):
            inputs = tokenizer(
                text, return_tensors="pt", truncation=True, max_length=max_length
            ).to(device)
            
            with torch.no_grad():
                outputs = model(**inputs, output_attentions=True)
            
            # outputs.attentions is tuple of (1, n_heads, seq_len, seq_len) per layer
            layer_entropies = []
            for layer_attn in outputs.attentions:
                # layer_attn: (1, n_heads, seq_len, seq_len)
                attn = layer_attn[0]  # (n_heads, seq_len, seq_len)
                
                # Compute entropy for each head: -sum(p * log(p)) averaged over query positions
                # Clamp to avoid log(0)
                attn_clamped = attn.clamp(min=1e-10)
                entropy = -(attn_clamped * attn_clamped.log()).sum(dim=-1)  # (n_heads, seq_len)
                mean_entropy = entropy.mean(dim=-1)  # (n_heads,)
                layer_entropies.append(mean_entropy.cpu().float().numpy())
            
            all_entropies.append(np.stack(layer_entropies, axis=0))  # (n_layers, n_heads)
            
            del outputs
            torch.cuda.empty_cache()
        
        return np.array(all_entropies)  # (n_texts, n_layers, n_heads)

    # Process one sample at a time (attention tensors: n_heads x seq_len x seq_len)
    # Use shorter max_length to keep memory manageable
    print("Processing legible CoTs...")
    legible_entropy = compute_attention_entropy(
        model, tokenizer, [c['cot_text'] for c in legible_sample], max_length=4096
    )

    print("Processing illegible CoTs...")
    illegible_entropy = compute_attention_entropy(
        model, tokenizer, [c['cot_text'] for c in illegible_sample], max_length=4096
    )

    print(f"Legible entropy shape: {legible_entropy.shape}")
    print(f"Illegible entropy shape: {illegible_entropy.shape}")

Processing legible CoTs...


Computing attention entropy:   0%|          | 0/28 [00:00<?, ?it/s]


ValueError: need at least one array to stack

In [ ]:
# Compare mean entropy per layer between legible and illegible
# Average across heads and samples
leg_mean_per_layer = legible_entropy.mean(axis=(0, 2))  # (n_layers,)
ill_mean_per_layer = illegible_entropy.mean(axis=(0, 2))  # (n_layers,)

# Bootstrap CI per layer
leg_ci = []
ill_ci = []
for l in range(legible_entropy.shape[1]):
    _, lo, hi = bootstrap_ci_metric(legible_entropy[:, l, :].mean(axis=1))
    leg_ci.append((lo, hi))
    _, lo, hi = bootstrap_ci_metric(illegible_entropy[:, l, :].mean(axis=1))
    ill_ci.append((lo, hi))

fig, ax = plt.subplots(figsize=(12, 5))
layers = range(len(leg_mean_per_layer))

leg_yerr = [[leg_mean_per_layer[l] - leg_ci[l][0] for l in layers],
            [leg_ci[l][1] - leg_mean_per_layer[l] for l in layers]]
ill_yerr = [[ill_mean_per_layer[l] - ill_ci[l][0] for l in layers],
            [ill_ci[l][1] - ill_mean_per_layer[l] for l in layers]]

ax.errorbar(list(layers), leg_mean_per_layer, yerr=leg_yerr,
            fmt='o-', capsize=3, label='Legible', alpha=0.8)
ax.errorbar(list(layers), ill_mean_per_layer, yerr=ill_yerr,
            fmt='s-', capsize=3, label='Illegible', alpha=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Attention Entropy')
ax.set_title('Exp A3: Attention Entropy -- Legible vs Illegible (G3)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(str(PHASE2_RESULTS_DIR / 'a3_attention_entropy.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Identify heads with largest legible-illegible entropy difference
# These are candidate "legibility heads"
diff = legible_entropy.mean(axis=0) - illegible_entropy.mean(axis=0)  # (n_layers, n_heads)

# Top 10 heads by absolute difference
flat_idx = np.argsort(np.abs(diff).flatten())[::-1]
print("Top 10 heads by |entropy difference| (legible - illegible):")
print(f"{'Layer':>6} {'Head':>6} {'Diff':>8} {'Leg Ent':>10} {'Ill Ent':>10}")
for i in range(min(10, len(flat_idx))):
    layer = flat_idx[i] // diff.shape[1]
    head = flat_idx[i] % diff.shape[1]
    print(f"{layer:>6} {head:>6} {diff[layer, head]:>+8.4f} "
          f"{legible_entropy[:, layer, head].mean():>10.4f} "
          f"{illegible_entropy[:, layer, head].mean():>10.4f}")

In [ ]:
# Heatmap of entropy difference across layers and heads
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(diff.T, aspect='auto', cmap='RdBu_r', vmin=-0.5, vmax=0.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Head')
ax.set_title('Attention Entropy Difference (Legible - Illegible)')
plt.colorbar(im, ax=ax, label='Entropy Diff')
fig.tight_layout()
fig.savefig(str(PHASE2_RESULTS_DIR / 'a3_attention_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save results and cleanup
results = {
    'legible_mean_entropy_per_layer': leg_mean_per_layer.tolist(),
    'illegible_mean_entropy_per_layer': ill_mean_per_layer.tolist(),
    'entropy_diff_shape': list(diff.shape),
    'n_legible': len(legible_sample),
    'n_illegible': len(illegible_sample),
}
with open(PHASE2_RESULTS_DIR / 'attention_results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Save the full entropy arrays for later analysis
np.save(PHASE2_RESULTS_DIR / 'legible_attention_entropy.npy', legible_entropy)
np.save(PHASE2_RESULTS_DIR / 'illegible_attention_entropy.npy', illegible_entropy)

if model is not None:
    del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("NB5 complete.")